# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hajergafsi/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup 
Each notebook runs in its own fresh Colab session, so I reconnect to the warehouse here. **On purpose, this query never touches the target/future window (`report_date >= 2026-03-15`) at all** — only feature-window data (`< 2026-03-15`) is ever pulled this week. That makes Section 4's leakage check trivial to prove: there is nothing from the future in this notebook to leak, because it was never queried.

In [2]:
import duckdb
import pandas as pd
import numpy as np
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("HF_TOKEN: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

MONTH = "2026-03"
FACT = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"
T = "2026-03-15"

signals = con.execute(f"""
    SELECT content_hash_id, client_hash_id,
           AVG(gsc_impressions)      AS avg_impressions,
           AVG(gsc_clicks)           AS avg_clicks,
           AVG(gsc_avg_position)     AS avg_position,
           AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) AS ctr,
           COUNT(DISTINCT report_date) AS days_seen
    FROM '{FACT}'
    WHERE report_date < DATE '{T}'
    GROUP BY 1, 2
""").df()

print("Shape:", signals.shape)
signals.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (319585, 7)


,content_hash_id,client_hash_id,avg_impressions,avg_clicks,avg_position,ctr,days_seen
0,content_93ae3d7b121fa641,client_73cda7b4e4f265ea,7.642857,0.0,13.002778,0.0,14
1,content_2ae3b7d9ff87c0cc,client_73cda7b4e4f265ea,5.928571,0.0,4.887103,0.0,14
2,content_0333fe1c5785e837,client_73cda7b4e4f265ea,2.214286,0.0,20.426667,0.0,14
3,content_654155d682f9e11c,client_73cda7b4e4f265ea,1.571429,0.0,75.824074,0.0,14
4,content_0edc7076d0aad1c2,client_73cda7b4e4f265ea,5.571429,0.0,15.509779,0.0,14


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal check 1 — Position vs CTR** (this is the signal behind FlyRank's real CTR-fix flag).

**Hypothesis:** pages ranking in a better position (a lower position number) should get a higher click-through rate — that's the whole premise behind flagging "low CTR at a good position" as a problem.

In [1]:
signals['position_tier'] = pd.cut(
    signals['avg_position'],
    bins=[0, 3, 10, 20, 1000],
    labels=['1-3', '4-10', '11-20', '21+']
)

position_ctr_table = signals.groupby('position_tier', observed=True).agg(
    n=('ctr', 'size'),
    avg_ctr=('ctr', 'mean')
).round(4)
print(position_ctr_table)

                   n  avg_ctr
position_tier                
1-3            16234   0.0092
4-10           69582   0.0045
11-20          26348   0.0034
21+            36946   0.0021


**Verdict (fill after running):** CONFIRMED `avg_ctr` clearly decreases as position tier gets worse (1-3 highest, 21+ lowest)

**Signal check 2 — Impression volume** (this is the signal behind FlyRank's real quick-win flag).

**Hypothesis:** a meaningful share of pages have enough volume to matter for prioritization — quick-win logic specifically targets higher-volume pages, since a fix there has more impact than fixing a page nobody sees.

In [6]:
signals['volume_tier'] = pd.cut(
    signals['avg_impressions'],
    bins=[-0.01, 5, 50, 500, 1e9],
    labels=['0-5', '6-50', '51-500', '500+']
)

volume_table = signals.groupby('volume_tier', observed=True).agg(
    n=('avg_impressions', 'size'),
    avg_ctr=('ctr', 'mean'),
    avg_position=('avg_position', 'mean')
).round(4)
print(volume_table)

                  n  avg_ctr  avg_position
volume_tier                               
0-5          236214   0.0065       18.8185
6-50          49960   0.0027       14.1815
51-500        30344   0.0032       10.8926
500+           3067   0.0028       13.6871


**Verdict (fill after running):** `MIXED` 
* Most pages are in the 0–5 impressions bucket, so volume is heavily skewed toward low-traffic pages.
* There is still a meaningful number of pages in the 51–500 bucket, which means a volume filter could help find pages worth reviewing.
* But the 500+ bucket is very small, so a rule that only targets very high-volume pages would miss most opportunities.
* The average position improves as volume increases, which makes sense: higher-volume pages tend to rank better. However, CTR does not meaningfully improve with volume, so volume alone is not a strong standalone signal for a CTR problem.

**My rule, in plain words:**

A page is a review candidate if it clears three bars at once: it gets enough traffic to matter (`avg_impressions >= 20`), it ranks well enough that clicks should be flowing (`avg_position` between 1 and 20), and despite that, its click-through rate is low (`ctr < 0.01`, i.e. under 1%). This is exactly the "good position, low CTR" pattern signal check 1 is testing — visible, well-ranked, but underperforming on clicks. Everything here is computed only from data before March 15 (the feature window) — nothing later is ever loaded.

**Reason code:** `low_ctr_visible_page` — the one and only reason code this rule can output (everything else gets `no_flag`).

**Action label:** `review_title_and_meta` — the suggested next step for a flagged page is to review its search title/meta description, since strong position with weak clicks usually points to a title/snippet mismatch rather than a ranking problem.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
def normalize(s):
    lo, hi = s.min(), s.max()
    if hi == lo:
        return s * 0
    return (s - lo) / (hi - lo)

# Score components -- built ONLY from feature-window signals already loaded above.
# Nothing here touches any future/target-window data (it was never queried this week).
volume_score   = normalize(np.log1p(signals['avg_impressions']))
position_score = normalize(-signals['avg_position'].fillna(signals['avg_position'].max()))
low_ctr_score  = normalize(-signals['ctr'].fillna(0))

signals['baseline_action_score'] = (
    0.40 * volume_score +
    0.35 * position_score +
    0.25 * low_ctr_score
)

# ONE reason code + ONE action label, from the plain-words rule above.
MIN_IMPRESSIONS = 20
GOOD_POSITION = 20
LOW_CTR = 0.01  # ctr here is a fraction (clicks/impressions), not the starter CSV's x100 percent style

is_candidate = (
    (signals['avg_impressions'] >= MIN_IMPRESSIONS) &
    (signals['avg_position'] > 0) & (signals['avg_position'] <= GOOD_POSITION) &
    (signals['ctr'] < LOW_CTR)
)

signals['reason_code'] = np.where(is_candidate, 'low_ctr_visible_page', 'no_flag')
signals['action'] = np.where(is_candidate, 'review_title_and_meta', 'monitor')

ranked = signals.sort_values('baseline_action_score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Wrote work/outputs/baseline_action_score.csv --", len(ranked), "rows")
print("Candidates flagged (low_ctr_visible_page):", (ranked['reason_code'] == 'low_ctr_visible_page').sum())
ranked.head(10)

Wrote work/outputs/baseline_action_score.csv -- 319585 rows
Candidates flagged (low_ctr_visible_page): 40687


,content_hash_id,client_hash_id,avg_impressions,avg_clicks,avg_position,ctr,days_seen,position_tier,volume_tier,baseline_action_score,reason_code,action
0,content_eadb33b5df496f4a,client_e547b89c05043229,10347.785714,151.500000,2.466116,0.014641,14,1-3,500+,0.993555,no_flag,monitor
1,content_ec2e0346994fb5a5,client_e547b89c05043229,8951.857143,57.285714,2.419526,0.006399,14,1-3,500+,0.989399,low_ctr_visible_page,review_title_and_meta
2,content_7172a7fad43f0998,client_62f4a7e64f5e0096,7351.214286,32.642857,3.030616,0.004440,14,4-10,500+,0.980676,low_ctr_visible_page,review_title_and_meta
3,content_e8a52cf3d5988c07,client_23a62021009f63c4,9913.785714,24.285714,16.041695,0.002450,14,11-20,500+,0.979422,low_ctr_visible_page,review_title_and_meta
4,content_b99ea6861864dea5,client_62f4a7e64f5e0096,6187.428571,12.214286,4.042024,0.001974,14,4-10,500+,0.972695,low_ctr_visible_page,review_title_and_meta
5,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,5983.571429,0.000000,5.794190,0.000000,14,4-10,500+,0.969761,low_ctr_visible_page,review_title_and_meta
6,content_7c6373141eae744a,client_62f4a7e64f5e0096,6011.428571,3.357143,5.854345,0.000558,14,4-10,500+,0.969754,low_ctr_visible_page,review_title_and_meta
7,content_e7b5dd4dff461ad2,client_08a6a72ff48e62c0,6029.214286,63.785714,4.544191,0.010579,14,4-10,500+,0.968856,no_flag,monitor
8,content_acbcc847f8996314,client_62f4a7e64f5e0096,5529.500000,8.571429,3.450064,0.001550,14,4-10,500+,0.968606,low_ctr_visible_page,review_title_and_meta
9,content_fd2117c2c6790e4b,client_73cda7b4e4f265ea,5272.642857,13.857143,3.651294,0.002628,14,4-10,500+,0.966051,low_ctr_visible_page,review_title_and_meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top20 = ranked.head(20)[['content_hash_id', 'client_hash_id', 'baseline_action_score',
                          'reason_code', 'action', 'avg_impressions', 'avg_position',
                          'ctr', 'days_seen']]
top20

,content_hash_id,client_hash_id,baseline_action_score,reason_code,action,avg_impressions,avg_position,ctr,days_seen
0,content_eadb33b5df496f4a,client_e547b89c05043229,0.993555,no_flag,monitor,10347.785714,2.466116,0.014641,14
1,content_ec2e0346994fb5a5,client_e547b89c05043229,0.989399,low_ctr_visible_page,review_title_and_meta,8951.857143,2.419526,0.006399,14
2,content_7172a7fad43f0998,client_62f4a7e64f5e0096,0.980676,low_ctr_visible_page,review_title_and_meta,7351.214286,3.030616,0.004440,14
3,content_e8a52cf3d5988c07,client_23a62021009f63c4,0.979422,low_ctr_visible_page,review_title_and_meta,9913.785714,16.041695,0.002450,14
4,content_b99ea6861864dea5,client_62f4a7e64f5e0096,0.972695,low_ctr_visible_page,review_title_and_meta,6187.428571,4.042024,0.001974,14
5,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,0.969761,low_ctr_visible_page,review_title_and_meta,5983.571429,5.794190,0.000000,14
6,content_7c6373141eae744a,client_62f4a7e64f5e0096,0.969754,low_ctr_visible_page,review_title_and_meta,6011.428571,5.854345,0.000558,14
7,content_e7b5dd4dff461ad2,client_08a6a72ff48e62c0,0.968856,no_flag,monitor,6029.214286,4.544191,0.010579,14
8,content_acbcc847f8996314,client_62f4a7e64f5e0096,0.968606,low_ctr_visible_page,review_title_and_meta,5529.500000,3.450064,0.001550,14
9,content_fd2117c2c6790e4b,client_73cda7b4e4f265ea,0.966051,low_ctr_visible_page,review_title_and_meta,5272.642857,3.651294,0.002628,14


**Top-20 review (fill one line per row, using the real printed table above):**

For each of rows 1-20, note: (a) the action (`review_title_and_meta` or `monitor`), (b) why it's there in plain words (e.g. "high volume + good position + very low CTR"), (c) a confidence note tied to `days_seen` (fewer observed days = lower confidence, since the average is noisier), and (d) what would make this pick wrong (e.g. "wrong if this page's low CTR is because the query intent doesn't match the page, not a fixable title problem" or "wrong if a sibling page absorbed this page's clicks — a consolidation case, not a CTR problem").

`<FILL with 20 short lines once you have the real top-20 table>`

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# WEAK PICKS -- pages with very few observed days have noisy averages; flag for a second look.
weak = ranked[(ranked['reason_code'] == 'low_ctr_visible_page') & (ranked['days_seen'] < 5)]
print("Flagged candidates built on fewer than 5 observed days (noisy, worth a second look):", len(weak))
weak.head(10)

Flagged candidates built on fewer than 5 observed days (noisy, worth a second look): 1


,content_hash_id,client_hash_id,avg_impressions,avg_clicks,avg_position,ctr,days_seen,position_tier,volume_tier,baseline_action_score,reason_code,action
48539,content_8a5f53dedcf19de6,client_400c21c81c8b46ef,24.0,0.0,11.208333,0.0,1,11-20,6-50,0.726621,low_ctr_visible_page,review_title_and_meta


**Leakage check for this baseline:**

- Every column used in the score (`avg_impressions`, `avg_clicks`, `avg_position`, `ctr`, `days_seen`) is computed only from `report_date < 2026-03-15` — the setup query in Section 0 never even requested rows on or after that date, so there is nothing from the future anywhere in this notebook.
- No FlyRank product flags (`health_score`, `priority_score`, `action_type`, refresh flags) were loaded or used — this rule is built entirely from raw observed search signals.
- No `label_declined` or any `tgt_*` column exists in this notebook at all — the future outcome from ML-04 was never pulled here, so there's nothing to leak by construction, not just by discipline.
- The weak picks above (fewer than 5 observed days) show where the score is most likely to be noisy rather than wrong — a human reviewer should treat these with lower confidence than picks built on a full two-week feature window.
- **A different kind of weakness worth naming:** this rule can't distinguish a real CTR problem from consolidation (a sibling page absorbing clicks) or from a genuine intent mismatch (the page ranks well for a query it doesn't actually answer) — both would produce the same "good position, low CTR" pattern without a title/meta fix being the right action.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.